In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import sys
import os

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from utils.bam import SymbolDataset, SymbolClassifier

In [ ]:
# Classifier Parameters
sf = 9
input = 64
hidden = 256
output = 2 ** sf
lr=1e-3
batch_size=32

folder_path = "classifier_dataset_sf{}_{}_{}".format(sf, input, output)

X = np.load(f"{folder_path}/X.npy")   # shape (30720, 16)
y = np.load(f"{folder_path}/y.npy")   # shape (30720,)

dataset = SymbolDataset(X, y)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

layers = [input,1024,512,output]
model = SymbolClassifier(layers).to(device)
criterion = nn.CrossEntropyLoss()   # Softmax included
optimizer = optim.Adam(model.parameters(), lr=lr) #weight_decay=1e-4 optim.Adam

In [4]:
num_epochs = 100

for epoch in range(num_epochs):
    model.train()

    total_loss = 0.0
    correct = 0
    total = 0

    for X_batch, y_batch in dataloader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        logits = model(X_batch)           ## (batch, 512)
        loss = criterion(logits, y_batch) ## match predicted with groundtruth class

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * X_batch.size(0)

        preds = torch.argmax(logits, dim=1)
        correct += (preds == y_batch).sum().item()
        total += y_batch.size(0)

    avg_loss = total_loss / total
    acc = correct / total * 100
    if epoch == 0:
        print("logits[0][:10]:", logits[0][:10])
        print("pred:", preds[0].item())
        print("gt:", y_batch[0].item())

    print(f"Epoch [{epoch+1}/{num_epochs}] "
          f"Loss: {avg_loss:.4f} | Accuracy: {acc:.2f}%")
    
torch.save(model.state_dict(), "symbol_classifier.pt")

Epoch [1/100] Loss: 3.8964 | Accuracy: 8.22%
Epoch [2/100] Loss: 2.8676 | Accuracy: 15.28%
Epoch [3/100] Loss: 2.5806 | Accuracy: 20.45%
Epoch [4/100] Loss: 2.3787 | Accuracy: 24.46%
Epoch [5/100] Loss: 2.2234 | Accuracy: 27.95%
Epoch [6/100] Loss: 2.0964 | Accuracy: 31.27%
Epoch [7/100] Loss: 1.9935 | Accuracy: 33.68%
Epoch [8/100] Loss: 1.8968 | Accuracy: 36.67%
Epoch [9/100] Loss: 1.8141 | Accuracy: 39.12%
Epoch [10/100] Loss: 1.7380 | Accuracy: 41.44%
Epoch [11/100] Loss: 1.6713 | Accuracy: 43.59%
Epoch [12/100] Loss: 1.6132 | Accuracy: 45.51%
Epoch [13/100] Loss: 1.5562 | Accuracy: 47.03%
Epoch [14/100] Loss: 1.4961 | Accuracy: 49.10%
Epoch [15/100] Loss: 1.4497 | Accuracy: 50.47%
Epoch [16/100] Loss: 1.4012 | Accuracy: 52.54%
Epoch [17/100] Loss: 1.3544 | Accuracy: 53.70%
Epoch [18/100] Loss: 1.3044 | Accuracy: 55.96%
Epoch [19/100] Loss: 1.2679 | Accuracy: 56.91%
Epoch [20/100] Loss: 1.2295 | Accuracy: 58.73%
Epoch [21/100] Loss: 1.1900 | Accuracy: 59.93%
Epoch [22/100] Loss: 1.

In [5]:

def evaluate(model, dataloader):
    model.eval()
    correct1 = 0
    correct2 = 0
    total = 0

    with torch.no_grad():
        for X_batch, y_batch in dataloader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            logits = model(X_batch)

            _, top2 = torch.topk(logits, k=2, dim=1)
            preds = torch.argmax(logits, dim=1)

            correct1 += (preds == y_batch).sum().item()
            correct2 += (top2 == y_batch.unsqueeze(1)).any(dim=1).sum().item()
            total += y_batch.size(0)

    print(f"Top-1 Accuracy: {100*correct1/total:.2f}%")
    print(f"Top-2 Accuracy: {100*correct2/total:.2f}%")


# Load

model.load_state_dict(torch.load("symbol_classifier.pt"))
model.eval()
evaluate(model,dataloader)

Top-1 Accuracy: 95.13%
Top-2 Accuracy: 99.60%
